In [ ]:
from datetime import date, timedelta
from IPython.display import display

import polars as pl
import numpy as np

from mars.analysis.profiler import MarsDataProfiler
from mars.feature.binner import MarsNativeBinner, MarsOptimalBinner
from mars.analysis.evaluator import MarsBinEvaluator
from mars.utils.logger import set_log_level

pl.Config.set_fmt_str_lengths(100)
set_log_level("WARNING")

In [ ]:
np.random.seed(2024)
N_MONTHS = 6        
N_SAMPLES = 20000     
BASE_DATE = date(2024, 1, 1)

data_parts = []

print(f"🛠️  正在生成 20 个特征的模拟数据 ({N_MONTHS} 个月 x {N_SAMPLES} 样本)...")

for i in range(N_MONTHS):
    curr_month = BASE_DATE + timedelta(days=31 * i)
    month_data = {
        "apply_date": [curr_month] * N_SAMPLES
    }
    
    # 初始 Logits 偏置
    # 设置为 -3.0，使基础坏账率保持在 4.7% 左右 (Sigmoid(-3) ≈ 0.047)
    total_logits = np.full(N_SAMPLES, -3.0) 
    
    # -------------------------------------------------
    # A组: [00-04] 稳定特征 (Stable)
    # -------------------------------------------------
    # 模拟信用评分、收入等级等强特征
    for k in range(5):
        feat_name = f"feat_stable_{k:02d}"
        raw_val = np.random.normal(0, 1, N_SAMPLES)
        
        # 降低系数 (0.3 ~ 0.5)，避免单特征 IV 爆炸
        coef = np.random.choice([0.3, -0.3, 0.5, -0.4])
        total_logits += coef * raw_val
        
        # 少量缺失 (1%)
        val_with_nan = raw_val.copy()
        mask_nan = np.random.rand(N_SAMPLES) < 0.01
        val_with_nan[mask_nan] = np.nan
        month_data[feat_name] = val_with_nan

    # -------------------------------------------------
    # B组: [05-09] 漂移特征 (Drift)
    # -------------------------------------------------
    # 模拟某些政策变动导致的客群偏移
    for k in range(5):
        feat_name = f"feat_drift_{k:02d}"
        
        # 漂移幅度温和一点：半年偏移 0.5 个标准差
        drift_amount = i * 0.1 
        raw_val = np.random.normal(loc=drift_amount, scale=1, size=N_SAMPLES)
        
        # 弱相关 (0.1)，即使漂移也不会导致 badrate 剧烈波动
        total_logits += 0.1 * raw_val 
        
        val_with_nan = raw_val.copy()
        val_with_nan[np.random.rand(N_SAMPLES) < 0.05] = np.nan
        month_data[feat_name] = val_with_nan

    # -------------------------------------------------
    # C组: [10-14] 缺失恶化特征 (MissInc)
    # -------------------------------------------------
    # 模拟三方数据接口不稳定性
    for k in range(5):
        feat_name = f"feat_miss_{k:02d}"
        raw_val = np.random.normal(0, 1, N_SAMPLES)
        
        # 这种特征通常还是有点用的
        total_logits += 0.2 * raw_val
        
        # 缺失率从 70% -> 95%
        miss_rate = 0.7 + (i * 0.05) 
        val_with_nan = raw_val.copy()
        val_with_nan[np.random.rand(N_SAMPLES) < miss_rate] = np.nan
        month_data[feat_name] = val_with_nan

    # -------------------------------------------------
    # D组: [15-19] 纯噪声特征 (Noise)
    # -------------------------------------------------
    for k in range(5):
        feat_name = f"feat_noise_{k:02d}"
        raw_val = np.random.normal(0, 2, N_SAMPLES)
        # 完全不加入 total_logits
        month_data[feat_name] = raw_val

    # -------------------------------------------------
    # 生成 Target
    # -------------------------------------------------
    # 增加随机噪声项，模拟模型无法解释的部分
    # scale=1.0 保证信噪比合理，坏账率不会非0即1
    total_logits += np.random.normal(0, 1.0, N_SAMPLES)
    
    probs = 1 / (1 + np.exp(-total_logits))
    target = (np.random.rand(N_SAMPLES) < probs).astype(int)
    
    month_data["target"] = target
    
    # 统计当月坏账率
    bad_rate = target.mean()
    print(f"  - Month {i+1}: Bad Rate = {bad_rate:.2%}")
    
    data_parts.append(pl.DataFrame(month_data))

# 合并
df_test: pl.DataFrame = pl.concat(data_parts)
df_test = df_test.with_columns(pl.col("apply_date").cast(pl.String))

print("-" * 50)
print(f"✅ 数据生成完毕! Shape: {df_test.shape}")
print(f"   全局坏账率: {df_test['target'].mean():.2%}")
print("-" * 50)



🛠️  正在生成 20 个特征的模拟数据 (6 个月 x 20000 样本)...
  - Month 1: Bad Rate = 8.95%
  - Month 2: Bad Rate = 9.46%
  - Month 3: Bad Rate = 9.83%
  - Month 4: Bad Rate = 10.03%
  - Month 5: Bad Rate = 10.49%
  - Month 6: Bad Rate = 10.72%
--------------------------------------------------
✅ 数据生成完毕! Shape: (120000, 22)
   全局坏账率: 9.91%
--------------------------------------------------


In [ ]:
df_test.head()

apply_date,feat_stable_00,feat_stable_01,feat_stable_02,feat_stable_03,feat_stable_04,feat_drift_00,feat_drift_01,feat_drift_02,feat_drift_03,feat_drift_04,feat_miss_00,feat_miss_01,feat_miss_02,feat_miss_03,feat_miss_04,feat_noise_00,feat_noise_01,feat_noise_02,feat_noise_03,feat_noise_04,target
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i32
"""2024-01-01""",1.668047,0.001636,-0.369792,-1.788306,0.456893,0.558442,0.572631,1.000406,1.312962,1.100726,NaN,-0.387719,0.186176,NaN,0.619361,-2.046308,2.272591,2.792012,1.007653,-1.36604,1
"""2024-01-01""",0.737348,0.43103,0.704325,-0.646752,-0.339895,1.272808,-1.481064,0.762491,0.156319,1.133006,NaN,NaN,NaN,NaN,NaN,0.437779,0.419399,-2.19845,-0.033999,0.046231,0
"""2024-01-01""",-0.201538,-0.269814,0.293191,-0.289382,-2.080532,-1.66749,-0.190527,0.252458,-1.884677,1.454692,-0.850332,NaN,-0.605657,0.679121,NaN,2.524236,1.556764,2.372679,1.951594,2.254981,0
"""2024-01-01""",-0.150912,0.107329,-1.788568,-0.313834,-1.39575,-0.911884,-0.805904,NaN,-0.496687,-0.669048,-0.200391,NaN,-0.591411,-0.085521,NaN,1.337186,-3.203901,-0.838905,0.146901,-3.287832,0
"""2024-01-01""",0.916052,-0.0137,0.071821,0.168971,1.053674,-0.145678,NaN,0.295431,0.491123,-0.367002,-0.302291,0.686308,NaN,NaN,NaN,1.685961,3.22031,-1.081754,-2.042914,-0.734279,0


In [ ]:
df_test.describe()

statistic,apply_date,feat_stable_00,feat_stable_01,feat_stable_02,feat_stable_03,feat_stable_04,feat_drift_00,feat_drift_01,feat_drift_02,feat_drift_03,feat_drift_04,feat_miss_00,feat_miss_01,feat_miss_02,feat_miss_03,feat_miss_04,feat_noise_00,feat_noise_01,feat_noise_02,feat_noise_03,feat_noise_04,target
str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""","""120000""",120000.0,120000.0,120000.0,120000.0,120000.0,120000.0,120000.0,120000.0,120000.0,120000.0,120000.0,120000.0,120000.0,120000.0,120000.0,120000.0,120000.0,120000.0,120000.0,120000.0,120000.0
"""null_count""","""0""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",null,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.001462,0.008271,0.001408,0.002392,0.002051,0.099125
"""std""",null,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.995679,1.998597,1.999741,1.994784,1.995384,0.298831
"""min""","""2024-01-01""",-4.659177,-4.390396,-4.407705,-4.102802,-3.951435,-3.914277,-4.287416,-4.170083,-4.036734,-4.782146,-3.607031,-4.103873,-4.358812,-3.635214,-4.25254,-8.283695,-8.520235,-8.633718,-8.501689,-8.904743,0.0
"""25%""",null,-0.668368,-0.665434,-0.673857,-0.669417,-0.670176,-0.402009,-0.397612,-0.395651,-0.390354,-0.393279,NaN,NaN,NaN,NaN,NaN,-1.349243,-1.335083,-1.354274,-1.346748,-1.348042,0.0
"""50%""",null,0.012339,0.008848,-0.000048,0.009665,0.010444,0.308003,0.314633,0.317925,0.31338,0.318583,NaN,NaN,NaN,NaN,NaN,0.000281,0.007665,0.008547,0.010019,0.003375,0.0
"""75%""",null,0.701091,0.693815,0.693036,0.692653,0.700509,1.062512,1.0648,1.062897,1.064598,1.066077,NaN,NaN,NaN,NaN,NaN,1.351206,1.358694,1.343728,1.351203,1.35264,0.0
"""max""","""2024-06-04""",4.402141,4.688606,5.088348,3.970648,4.664161,4.970171,4.465295,4.50833,4.350508,4.745254,4.126921,4.216725,3.936684,4.200971,4.314615,8.430073,8.437149,8.165473,8.71499,9.245517,1.0


In [ ]:
features = [col for col in df_test.columns if col.startswith("feat_")]
len(features)

20

In [ ]:
test_profiler = MarsDataProfiler(
    df=df_test,
    features=features,
    exclude_features = None,
    include_dtypes = None,
    custom_missing_values = None,
    custom_special_values = None,
    overview_batch_size = 500,
    psi_batch_size = 50,
    psi_n_bins = 10,
    psi_bin_method = "quantile",
    psi_cv_ignore_threshold = 0.05,
    sample_frac = None,
    config = None
)

In [ ]:
test_profiler_report = test_profiler.generate_profile(
    profile_by="month",
    dt_col="apply_date",
    config_overrides={
        "dq_metrics": ["missing", "zeros", "top1"],
        "stat_metrics": ["mean", "std", "min", "max",]
    }
)
test_profiler_report

In [ ]:
test_profiler_report.show_overview()

feature,dtype,distribution,missing_rate,zeros_rate,top1_ratio,top1_value,mean,std,min,max
feat_drift_00,Float64,▂▂▄█▆▃▂▂,5.01%,0.00%,5.01%,NaN,0.24,1.01,-3.91,4.97
feat_drift_01,Float64,▂▂▃▇█▄▂▂,4.99%,0.00%,4.99%,NaN,0.25,1.02,-4.29,4.47
feat_drift_02,Float64,▂▂▃▇█▄▂▂,4.90%,0.00%,4.90%,NaN,0.25,1.01,-4.17,4.51
feat_drift_03,Float64,▂▂▃▇█▄▂▂,5.01%,0.00%,5.01%,NaN,0.25,1.01,-4.04,4.35
feat_drift_04,Float64,▂▂▂▆█▄▂▂,5.00%,0.00%,5.00%,NaN,0.25,1.01,-4.78,4.75
feat_miss_00,Float64,▂▂▅█▆▃▂▂,82.45%,0.00%,82.45%,NaN,-0.00,1.00,-3.61,4.13
feat_miss_01,Float64,▂▂▄█▇▄▂▂,82.56%,0.00%,82.56%,NaN,-0.00,1.00,-4.10,4.22
feat_miss_02,Float64,▂▂▃▆█▄▂▂,82.48%,0.00%,82.48%,NaN,-0.00,1.00,-4.36,3.94
feat_miss_03,Float64,▂▂▅█▆▃▂▂,82.33%,0.00%,82.33%,NaN,0.00,1.00,-3.64,4.20
feat_miss_04,Float64,▂▂▄█▇▄▂▂,82.61%,0.00%,82.61%,NaN,0.01,1.01,-4.25,4.31


In [ ]:
test_profiler_report.show_dq("missing", ascending=False)

feature,dtype,2024-06-01,2024-05-01,2024-04-01,2024-03-01,2024-02-01,2024-01-01,total
feat_drift_00,Float64,5.16%,5.22%,4.84%,4.78%,5.13%,4.93%,5.01%
feat_drift_01,Float64,4.94%,4.98%,5.15%,4.92%,5.01%,4.94%,4.99%
feat_drift_02,Float64,4.58%,4.98%,5.16%,5.02%,4.98%,4.67%,4.90%
feat_drift_03,Float64,5.05%,5.12%,5.07%,5.22%,4.70%,4.91%,5.01%
feat_drift_04,Float64,4.93%,5.13%,5.03%,4.92%,4.95%,5.02%,5.00%
feat_miss_00,Float64,95.03%,89.78%,84.80%,80.03%,75.11%,69.95%,82.45%
feat_miss_01,Float64,94.67%,90.36%,85.30%,80.12%,74.83%,70.05%,82.56%
feat_miss_02,Float64,95.06%,90.00%,84.76%,79.93%,74.74%,70.41%,82.48%
feat_miss_03,Float64,94.89%,89.50%,85.06%,80.09%,74.31%,70.10%,82.33%
feat_miss_04,Float64,94.95%,89.82%,85.08%,80.54%,75.27%,70.00%,82.61%


In [ ]:
test_profiler_report.show_trend("mean")

feature,dtype,2024-01-01,2024-02-01,2024-03-01,2024-04-01,2024-05-01,2024-06-01,total,group_mean,group_var,group_cv
feat_drift_00,Float64,-0.02,0.09,0.20,0.30,0.40,0.49,0.24,0.24,0.0361,0.7829
feat_drift_01,Float64,0.00,0.09,0.21,0.31,0.39,0.50,0.25,0.25,0.0345,0.7452
feat_drift_02,Float64,-0.01,0.10,0.21,0.30,0.39,0.50,0.25,0.25,0.0353,0.7560
feat_drift_03,Float64,-0.01,0.11,0.19,0.30,0.41,0.49,0.25,0.25,0.0355,0.7548
feat_drift_04,Float64,0.01,0.10,0.20,0.30,0.40,0.50,0.25,0.25,0.0346,0.7364
feat_miss_00,Float64,-0.01,0.01,0.00,0.01,-0.02,0.03,-0.00,0.00,0.0003,20.0575
feat_miss_01,Float64,-0.01,-0.00,0.02,-0.03,-0.00,0.00,-0.00,-0.00,0.0002,4.4634
feat_miss_02,Float64,-0.01,0.02,0.00,-0.00,0.00,-0.04,-0.00,-0.01,0.0004,4.1226
feat_miss_03,Float64,-0.00,0.01,-0.00,0.00,0.02,0.03,0.00,0.01,0.0002,1.6471
feat_miss_04,Float64,0.01,-0.00,0.01,0.02,0.01,0.08,0.01,0.02,0.0009,1.3866


In [ ]:
overview, dq_table, stat_table = test_profiler_report.get_profile_data()
display(overview.head(), dq_table["missing"].head())

feature,dtype,distribution,missing_rate,zeros_rate,top1_ratio,top1_value,mean,std,min,max
str,str,str,f64,f64,f64,str,f64,f64,f64,f64
"""feat_drift_00""","""Float64""","""▂▂▄█▆▃▂▂""",0.050125,0.0,0.050125,"""NaN""",0.242571,1.01432,-3.914277,4.970171
"""feat_drift_01""","""Float64""","""▂▂▃▇█▄▂▂""",0.049892,0.0,0.049892,"""NaN""",0.249165,1.016231,-4.287416,4.465295
"""feat_drift_02""","""Float64""","""▂▂▃▇█▄▂▂""",0.049008,0.0,0.049008,"""NaN""",0.248441,1.014433,-4.170083,4.50833
"""feat_drift_03""","""Float64""","""▂▂▃▇█▄▂▂""",0.050133,0.0,0.050133,"""NaN""",0.249488,1.011788,-4.036734,4.350508
"""feat_drift_04""","""Float64""","""▂▂▂▆█▄▂▂""",0.049983,0.0,0.049983,"""NaN""",0.252674,1.012562,-4.782146,4.745254


feature,dtype,2024-01-01,2024-02-01,2024-03-01,2024-04-01,2024-05-01,2024-06-01,total
str,str,f64,f64,f64,f64,f64,f64,f64
"""feat_drift_00""","""Float64""",0.04935,0.05135,0.0478,0.04845,0.05225,0.05155,0.050125
"""feat_drift_01""","""Float64""",0.0494,0.0501,0.04925,0.05145,0.04975,0.0494,0.049892
"""feat_drift_02""","""Float64""",0.04675,0.04985,0.0502,0.0516,0.04985,0.0458,0.049008
"""feat_drift_03""","""Float64""",0.0491,0.04695,0.05225,0.0507,0.05125,0.05055,0.050133
"""feat_drift_04""","""Float64""",0.0502,0.04945,0.04925,0.0503,0.05135,0.04935,0.049983


In [ ]:
(
    overview
    .filter(pl.col("missing_rate") < 0.8)
    .select("feature")
    .to_series()
    .to_list()
)

['feat_drift_00',
 'feat_drift_01',
 'feat_drift_02',
 'feat_drift_03',
 'feat_drift_04',
 'feat_noise_00',
 'feat_noise_01',
 'feat_noise_02',
 'feat_noise_03',
 'feat_noise_04',
 'feat_stable_00',
 'feat_stable_01',
 'feat_stable_02',
 'feat_stable_03',
 'feat_stable_04']

In [ ]:
test_native_binner = MarsNativeBinner(
    features=features,
    method = "quantile",
    n_bins = 10,
    special_values = None,
    missing_values = None,
    min_samples = 0.05, # 仅对 cart 有效
    cart_params = None,
    remove_empty_bins = False, # 仅对 uniform 有效
    n_jobs = -1
)
test_native_binner.fit(df_test, df_test["target"])

,features,"['feat_stable_00', 'feat_stable_01', ...]"
,method,'quantile'
,n_bins,10
,special_values,[]
,missing_values,[]
,min_samples,0.05
,cart_params,None
,remove_empty_bins,False
,n_jobs,23


In [ ]:
trans_cols = ["feat_stable_00", "feat_drift_00", "feat_miss_00", "feat_noise_00"]
display(test_native_binner.transform(df_test[trans_cols], return_type ="label", lazy=True).collect().head(5))
# display(test_native_binner.transform(df_test[trans_cols], return_type ="index").head(5))
display(test_native_binner.transform(df_test[trans_cols], return_type ="woe").head(5))

feat_stable_00,feat_drift_00,feat_miss_00,feat_noise_00,feat_stable_00_bin,feat_noise_00_bin,feat_drift_00_bin,feat_miss_00_bin
f64,f64,f64,f64,str,str,str,str
1.668047,0.558442,NaN,-2.046308,"""09_[1.29, inf)""","""01_[-2.55, -1.68)""","""06_[0.5, 0.778)""","""Missing"""
0.737348,1.272808,NaN,0.437779,"""07_[0.526, 0.842)""","""05_[0.000281, 0.51)""","""08_[1.1, 1.54)""","""Missing"""
-0.201538,-1.66749,-0.850332,2.524236,"""04_[-0.255, 0.0005)""","""08_[1.68, 2.55)""","""00_[-inf, -1.05)""","""02_[-0.854, -0.534)"""
-0.150912,-0.911884,-0.200391,1.337186,"""04_[-0.255, 0.0005)""","""07_[1.05, 1.68)""","""01_[-1.05, -0.615)""","""04_[-0.252, 0.000196)"""
0.916052,-0.145678,-0.302291,1.685961,"""08_[0.842, 1.29)""","""08_[1.68, 2.55)""","""03_[-0.293, -0.0174)""","""03_[-0.534, -0.252)"""


feat_stable_00,feat_drift_00,feat_miss_00,feat_noise_00,feat_stable_00_woe,feat_noise_00_woe,feat_drift_00_woe,feat_miss_00_woe
f64,f64,f64,f64,f32,f32,f32,f32
1.668047,0.558442,NaN,-2.046308,0.087631,0.026317,0.000073,0.009086
0.737348,1.272808,NaN,0.437779,-0.004348,0.04275,0.151691,0.009086
-0.201538,-1.66749,-0.850332,2.524236,-0.001514,-0.093493,-0.155635,-0.163249
-0.150912,-0.911884,-0.200391,1.337186,-0.001514,-0.024026,-0.102276,-0.10398
0.916052,-0.145678,-0.302291,1.685961,-0.00057,-0.093493,-0.042901,-0.133252


In [ ]:
test_native_bin_performance_table = test_native_binner.profile_bin_performance(df_test[features], df_test["target"], update_woe=True)

In [ ]:
trans_cols = ["feat_stable_00", "feat_drift_00", "feat_miss_00", "feat_noise_00"]
display(test_native_binner.transform(df_test[trans_cols], return_type ="label", lazy=True).collect().head(5))
# display(test_native_binner.transform(df_test[trans_cols], return_type ="index").head(5))
display(test_native_binner.transform(df_test[trans_cols], return_type ="woe").head(5))

feat_stable_00,feat_drift_00,feat_miss_00,feat_noise_00,feat_stable_00_bin,feat_noise_00_bin,feat_drift_00_bin,feat_miss_00_bin
f64,f64,f64,f64,str,str,str,str
1.668047,0.558442,NaN,-2.046308,"""09_[1.29, inf)""","""01_[-2.55, -1.68)""","""06_[0.5, 0.778)""","""Missing"""
0.737348,1.272808,NaN,0.437779,"""07_[0.526, 0.842)""","""05_[0.000281, 0.51)""","""08_[1.1, 1.54)""","""Missing"""
-0.201538,-1.66749,-0.850332,2.524236,"""04_[-0.255, 0.0005)""","""08_[1.68, 2.55)""","""00_[-inf, -1.05)""","""02_[-0.854, -0.534)"""
-0.150912,-0.911884,-0.200391,1.337186,"""04_[-0.255, 0.0005)""","""07_[1.05, 1.68)""","""01_[-1.05, -0.615)""","""04_[-0.252, 0.000196)"""
0.916052,-0.145678,-0.302291,1.685961,"""08_[0.842, 1.29)""","""08_[1.68, 2.55)""","""03_[-0.293, -0.0174)""","""03_[-0.534, -0.252)"""


feat_stable_00,feat_drift_00,feat_miss_00,feat_noise_00,feat_stable_00_woe,feat_noise_00_woe,feat_drift_00_woe,feat_miss_00_woe
f64,f64,f64,f64,f32,f32,f32,f32
1.668047,0.558442,NaN,-2.046308,0.087631,0.026317,0.000073,0.009086
0.737348,1.272808,NaN,0.437779,-0.004348,0.04275,0.151691,0.009086
-0.201538,-1.66749,-0.850332,2.524236,-0.001514,-0.093493,-0.155635,-0.163249
-0.150912,-0.911884,-0.200391,1.337186,-0.001514,-0.024026,-0.102276,-0.10398
0.916052,-0.145678,-0.302291,1.685961,-0.00057,-0.093493,-0.042901,-0.133252


In [ ]:
test_opt_biner = MarsOptimalBinner(
    features=features,
    cat_features = None,
    n_bins = 10,
    min_n_bins = 2,
    min_bin_size = 0.02,
    n_prebins = 20,
    prebinning_method = "cart",
    min_prebin_size = 0.01,
    monotonic_trend = "auto_asc_desc",
    solver = "cp",
    time_limit = 2,
    special_values = None,
    missing_values = None,
    cart_params = {"random_state": 2025},
    n_jobs = -1
)
test_opt_biner.fit(df_test, df_test["target"])

In [ ]:
test_opt_biner.fit_failures_

In [ ]:
trans_cols = ["feat_stable_00", "feat_drift_00", "feat_miss_00", "feat_noise_00"]
display(test_opt_biner.transform(df_test[trans_cols], return_type ="label", lazy=True).collect().head(3))
display(test_opt_biner.transform(df_test[trans_cols], return_type ="index").head(3))
display(test_opt_biner.transform(df_test[trans_cols], return_type ="woe").head(3))

In [ ]:
test_opt_bin_performance_table = test_opt_biner.profile_bin_performance(df_test[features], df_test["target"])
test_opt_bin_performance_table

In [ ]:
test_evaluator = MarsBinEvaluator(binner=test_native_binner, target_col="target")

test_evaluator_report = test_evaluator.evaluate(
    df=df_test,
    features=features,
    profile_by="month",
    dt_col="apply_date"
)

In [ ]:
test_evaluator_report

In [ ]:
display(test_evaluator_report.show_trend("psi", ascending=False))

In [ ]:
display(test_evaluator_report.show_trend("iv", ascending=True))

In [ ]:
display(test_evaluator_report.show_trend("risk_corr", ascending=False))

In [ ]:
summary_df, trend_tables, detail_table = test_evaluator_report.get_evaluation_data()

In [ ]:
trend_tables["auc"]

In [ ]:
detail_table

In [ ]:
test_evaluator_report.write_excel()

In [ ]:
test_evaluator.plot_feature_binning_risk_trends(
    test_evaluator_report,
    sort_by="iv",
    dpi=144
)